<div align="right"><sub><b>Created by Ram Lakhan Meena, INSPECTOR CBIC</b><br>13-Oct-2026</sub></div>

# 🗂️ Drive File Manager for Google Colab — v2

Browse your Google Drive with **dropdowns**, **upload** from your computer (into an existing or a **new folder**),
and **download directly to your computer** — including Google Docs / Sheets / Slides.

### ▶️ How to use (top to bottom)
| Step | Cell | What it does |
|---|---|---|
| 1 | **① SETUP** | Connects Drive (approve the popup once). Set your start folder here. |
| 2 | **② PICKER** | Dropdowns to move between folders, select files, create a new folder, search. |
| 3 | **③ UPLOAD** | Pick files from your computer → saved in the picker's folder (or a new sub-folder). |
| 4 | **④ DOWNLOAD** | Downloads what you selected in ② straight to your computer. |
| – | ⑤ – ⑧ | Quick download by name · Colab workspace ⇄ Drive · Folder report · Disconnect. |

> Tip: click **Runtime → Run all** once, or run ① and ② and keep the picker open while you use ③ / ④.

### 📝 Version log
| Version | Date | Changes |
|---|---|---|
| v1 | before 13-Oct-2026 | Original connector: mount, list, copy to `/content`, upload helpers, export by name, zip-folder upload. |
| **v2** | **13-Oct-2026** | Full reorganisation into 8 numbered form cells (code hidden). **New dropdown picker** (Drive / sub-folder dropdowns, multi-select file list, Up / Home / Refresh, search, *open in Drive* link). **Fixed direct download**: Google Docs/Sheets/Slides/Drawings were being downloaded as useless `.gdoc/.gsheet` pointer files — they are now exported to Office or PDF (file ID read from the mount, no more guessing by name; export-link fallback for files > 10 MB). Regular files are checksum-verified, with Drive-API fallback if the mount read fails. Several files / whole folders download as one ZIP (Google files inside are exported too). **New folder** creation (picker button or upload option). Upload with *keep both / overwrite / skip* and optional auto-unzip. Fixed the f-string syntax error that broke exports on older Python. Removed duplicate mount cells. |

## ① Setup

In [ ]:
#@title ① SETUP — run this first (connects Google Drive and loads the tool) { display-mode: "form" }
#@markdown **Folder the picker should open in** (inside *My Drive*; created if missing):
START_FOLDER = "File for Working in"  #@param {type:"string"}

import sys, os, io, re, json, time, shutil, hashlib, zipfile, datetime, fnmatch

if 'google.colab' not in sys.modules:
    raise RuntimeError("This notebook only works inside Google Colab (colab.research.google.com).")

from google.colab import drive, auth, files as colab_files
from IPython.display import display, HTML
import ipywidgets as W

# ───────────────────────── paths ─────────────────────────
MOUNT_POINT    = '/content/drive'
MY_DRIVE       = os.path.join(MOUNT_POINT, 'MyDrive')
SHARED_DRIVES  = os.path.join(MOUNT_POINT, 'Shareddrives')
WORK_DIR       = '/content/_drive_tool'
DL_DIR         = os.path.join(WORK_DIR, 'downloads')     # files staged here before the browser download
TMP_DIR        = os.path.join(WORK_DIR, 'tmp')
FROM_DRIVE_DIR = '/content/from_drive'                   # "copy to Colab workspace" target

if not os.path.isdir(MY_DRIVE):
    drive.mount(MOUNT_POINT)
for _d in (DL_DIR, TMP_DIR):
    os.makedirs(_d, exist_ok=True)

# ───────────────────── Google-native files ─────────────────────
# On the mounted Drive, Google Docs/Sheets/Slides are only tiny ".gdoc/.gsheet/.gslides" pointer files.
# Copying or downloading those pointers gives a useless file — they must be EXPORTED through the Drive API.
_PDF  = 'application/pdf'
NATIVE = {
    '.gdoc':    {'type': 'Google Doc',     'Office': ('docx', 'application/vnd.openxmlformats-officedocument.wordprocessingml.document'), 'PDF': ('pdf', _PDF)},
    '.gsheet':  {'type': 'Google Sheet',   'Office': ('xlsx', 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'),      'PDF': ('pdf', _PDF)},
    '.gslides': {'type': 'Google Slides',  'Office': ('pptx', 'application/vnd.openxmlformats-officedocument.presentationml.presentation'), 'PDF': ('pdf', _PDF)},
    '.gdraw':   {'type': 'Google Drawing', 'Office': ('png',  'image/png'),                                                               'PDF': ('pdf', _PDF)},
}
NOT_EXPORTABLE = {'.gform': 'Google Form', '.gsite': 'Google Site', '.gmap': 'Google My Map',
                  '.gtable': 'Google Table', '.gjam': 'Jamboard', '.gscript': 'Apps Script', '.glink': 'Drive link'}

_ICONS = {'.pdf': '📕', '.xlsx': '📊', '.xls': '📊', '.xlsm': '📊', '.csv': '📊', '.gsheet': '📊',
          '.doc': '📝', '.docx': '📝', '.gdoc': '📝', '.txt': '📝', '.ppt': '📽️', '.pptx': '📽️', '.gslides': '📽️',
          '.jpg': '🖼️', '.jpeg': '🖼️', '.png': '🖼️', '.gif': '🖼️', '.zip': '🗜️', '.rar': '🗜️', '.7z': '🗜️',
          '.ipynb': '📓', '.py': '🐍', '.mp4': '🎬', '.mp3': '🎵', '.json': '🔧', '.db': '🗄️', '.sqlite': '🗄️'}

def ext_of(p):        return os.path.splitext(p)[1].lower()
def is_native(p):     return ext_of(p) in NATIVE or ext_of(p) in NOT_EXPORTABLE
def icon_of(p):       return '📁' if os.path.isdir(p) else _ICONS.get(ext_of(p), '📄')

def human(n):
    for unit in ('B', 'KB', 'MB', 'GB'):
        if n < 1024 or unit == 'GB':
            return f"{n:.0f} {unit}" if unit == 'B' else f"{n:.1f} {unit}"
        n /= 1024

def md5_file(p):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def unique_path(p):
    """'a.pdf' -> 'a (1).pdf' -> 'a (2).pdf' … if the name is already taken."""
    if not os.path.exists(p):
        return p
    base, ext = os.path.splitext(p)
    if os.path.isdir(p):
        base, ext = p, ''
    i = 1
    while os.path.exists(f"{base} ({i}){ext}"):
        i += 1
    return f"{base} ({i}){ext}"

def safe_name(name):
    name = re.sub(r'[\x00-\x1f/\\]', '-', (name or '').strip())
    if name in ('', '.', '..'):
        raise ValueError("Please type a valid name.")
    return name

def rel(p):
    """Short display path: 'My Drive › A › B'."""
    p = os.path.normpath(p)
    if p.startswith(MY_DRIVE):
        tail = os.path.relpath(p, MY_DRIVE)
        return 'My Drive' + ('' if tail == '.' else ' › ' + tail.replace(os.sep, ' › '))
    if p.startswith(SHARED_DRIVES):
        return 'Shared drives › ' + os.path.relpath(p, SHARED_DRIVES).replace(os.sep, ' › ')
    return p

def folder_size(p):
    total = 0
    for r, _, fs in os.walk(p):
        for f in fs:
            try: total += os.path.getsize(os.path.join(r, f))
            except OSError: pass
    return total

# ───────────────────────── Drive API (only used for Google-native files & fallbacks) ─────────────────────────
_SVC = None
def drive_api():
    global _SVC
    if _SVC is None:
        print("🔐 One-time Google sign-in (needed to export Google Docs/Sheets/Slides). Approve the popup if it appears…")
        auth.authenticate_user()
        from googleapiclient.discovery import build
        _SVC = build('drive', 'v3', cache_discovery=False)
    return _SVC

def drive_id(path):
    """File ID straight from the mounted file (Colab stores it as an extended attribute) — no name guessing."""
    try:
        return os.getxattr(path, 'user.drive.id').decode().strip() or None
    except Exception:
        pass
    try:  # some pointer files contain JSON like {"doc_id": "..."}
        if os.path.isfile(path) and os.path.getsize(path) < 8192:
            with open(path, encoding='utf-8', errors='ignore') as f:
                d = json.load(f)
            v = d.get('doc_id') or d.get('resource_id', '')
            return v.split(':')[-1] or None
    except Exception:
        pass
    return None

def folder_link(path):
    fid = drive_id(path)
    return f"https://drive.google.com/drive/folders/{fid}" if fid else None

def resolve_id(path):
    fid = drive_id(path)
    if fid:
        return fid
    name = os.path.basename(path)
    if is_native(path):
        name = os.path.splitext(name)[0]
    q_name = name.replace('\\', '\\\\').replace("'", "\\'")
    q = f"name = '{q_name}' and trashed = false"
    parent = drive_id(os.path.dirname(path))
    if parent:
        q += f" and '{parent}' in parents"
    found = drive_api().files().list(q=q, fields='files(id,name,mimeType)', pageSize=10,
                                     supportsAllDrives=True, includeItemsFromAllDrives=True).execute().get('files', [])
    if is_native(path):
        found = [f for f in found if f['mimeType'].startswith('application/vnd.google-apps')] or found
    if not found:
        raise FileNotFoundError(f"Drive could not find '{name}'.")
    if len(found) > 1:
        print(f"   ⚠️ {len(found)} files are called '{name}'; using the first one.")
    return found[0]['id']

def _stream_to_file(request, out_path):
    from googleapiclient.http import MediaIoBaseDownload
    with io.FileIO(out_path, 'wb') as fh:
        dl, done = MediaIoBaseDownload(fh, request, chunksize=8 << 20), False
        while not done:
            _, done = dl.next_chunk()

def export_native(path, fmt, out_dir):
    """Export a Google Doc/Sheet/Slides/Drawing to Office or PDF. fmt = 'Office' | 'PDF'."""
    ext = ext_of(path)
    if ext in NOT_EXPORTABLE:
        raise ValueError(f"{NOT_EXPORTABLE[ext]} files cannot be downloaded — open them in Drive instead.")
    out_ext, mime = NATIVE[ext][fmt]
    fid = resolve_id(path)
    out = unique_path(os.path.join(out_dir, f"{os.path.splitext(os.path.basename(path))[0]}.{out_ext}"))
    svc = drive_api()
    from googleapiclient.errors import HttpError
    try:
        _stream_to_file(svc.files().export_media(fileId=fid, mimeType=mime), out)
    except HttpError as e:
        if os.path.exists(out): os.remove(out)
        if 'exportSizeLimitExceeded' not in str(e) and getattr(e, 'resp', None) is not None and e.resp.status != 403:
            raise
        # Large files (> 10 MB export limit): use the export link instead
        url = svc.files().get(fileId=fid, fields='exportLinks', supportsAllDrives=True).execute().get('exportLinks', {}).get(mime)
        if not url:
            raise
        import google.auth
        from google.auth.transport.requests import AuthorizedSession
        creds, _ = google.auth.default()
        r = AuthorizedSession(creds).get(url, stream=True)
        r.raise_for_status()
        with open(out, 'wb') as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
    return out

def copy_regular(src, out_dir):
    """Copy a normal Drive file out of the mount (checksum-verified); fall back to the Drive API if the mount read fails."""
    dst = unique_path(os.path.join(out_dir, os.path.basename(src)))
    try:
        shutil.copyfile(src, dst)
        if md5_file(src) != md5_file(dst):
            raise IOError('checksum mismatch')
    except Exception as e:
        print(f"   ↪ Mount copy failed ({e}); fetching through the Drive API instead…")
        if os.path.exists(dst): os.remove(dst)
        _stream_to_file(drive_api().files().get_media(fileId=resolve_id(src), supportsAllDrives=True), dst)
    return dst

def fetch_file(path, fmt, out_dir):
    return export_native(path, fmt, out_dir) if is_native(path) else copy_regular(path, out_dir)

def zip_items(paths, zip_path, fmt):
    """ZIP files and folders. Google-native files inside are exported (not zipped as useless pointers)."""
    skipped, count = [], 0
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
        def add(full, arc):
            nonlocal count
            if ext_of(full) in NOT_EXPORTABLE:
                skipped.append(arc); return
            if is_native(full):
                tmp = export_native(full, fmt, TMP_DIR)
                arc = os.path.splitext(arc)[0] + os.path.splitext(tmp)[1]
                zf.write(tmp, arc); os.remove(tmp)
            else:
                try:
                    zf.write(full, arc)
                except OSError:
                    tmp = copy_regular(full, TMP_DIR); zf.write(tmp, arc); os.remove(tmp)
            count += 1
            print(f"   + {arc}")
        for p in paths:
            if os.path.isdir(p):
                base = os.path.dirname(os.path.normpath(p))
                for r, ds, fs in os.walk(p):
                    ds.sort()
                    if not fs and not ds:
                        zf.writestr(os.path.relpath(r, base) + '/', '')
                    for f in sorted(fs):
                        add(os.path.join(r, f), os.path.relpath(os.path.join(r, f), base))
            else:
                add(p, os.path.basename(p))
    if skipped:
        print(f"   ⚠️ Skipped {len(skipped)} item(s) that Google does not allow to download (Forms, Sites…): {', '.join(skipped[:5])}")
    return zip_path, count

def _clean_old_downloads(minutes=15):
    now = time.time()
    for f in os.listdir(DL_DIR):
        p = os.path.join(DL_DIR, f)
        try:
            if now - os.path.getmtime(p) > minutes * 60:
                shutil.rmtree(p) if os.path.isdir(p) else os.remove(p)
        except OSError:
            pass

def browser_download(local_path):
    size = os.path.getsize(local_path)
    print(f"⬇️  Sending to your browser: {os.path.basename(local_path)}  ({human(size)})")
    if size > 1.5 * 1024 ** 3:
        print("   ⚠️ Very large file — the browser download may be slow or fail; downloading from drive.google.com is safer.")
    colab_files.download(local_path)

def download_items(paths, fmt='Office', several='One ZIP file'):
    """THE fixed direct download: Drive → (export / verified copy) → your computer."""
    paths = [p for p in paths if p]
    if not paths:
        print("⚠️ Nothing selected. In the picker (cell ②) click file(s) — Ctrl/Shift-click for several — then run this cell again.")
        return []
    _clean_old_downloads()
    stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    run_dir = unique_path(os.path.join(DL_DIR, stamp)); os.makedirs(run_dir)
    out = []
    if len(paths) == 1 and not os.path.isdir(paths[0]):
        out.append(fetch_file(paths[0], fmt, run_dir))
    elif len(paths) == 1:
        z = os.path.join(run_dir, f"{os.path.basename(os.path.normpath(paths[0]))}.zip")
        print(f"🗜️ Zipping folder '{os.path.basename(paths[0])}'…")
        out.append(zip_items(paths, unique_path(z), fmt)[0])
    elif several.startswith('One ZIP'):
        print(f"🗜️ Zipping {len(paths)} items…")
        out.append(zip_items(paths, os.path.join(run_dir, f"Drive_download_{stamp}.zip"), fmt)[0])
    else:
        for p in paths:
            if os.path.isdir(p):
                out.append(zip_items([p], unique_path(os.path.join(run_dir, os.path.basename(p) + '.zip')), fmt)[0])
            elif ext_of(p) in NOT_EXPORTABLE:
                print(f"   ⚠️ Skipped {os.path.basename(p)} ({NOT_EXPORTABLE[ext_of(p)]} cannot be downloaded)")
            else:
                out.append(fetch_file(p, fmt, run_dir))
    for i, f in enumerate(out):
        browser_download(f)
        if i < len(out) - 1:
            time.sleep(1.5)   # gives the browser time between downloads
    if len(out) > 1:
        print("ℹ️ If only one file arrived, allow 'multiple downloads' for colab.research.google.com in your browser.")
    print("✅ Done. Check your browser's Downloads folder.")
    return out

# ───────────────────────── upload & folders ─────────────────────────
def create_folder(parent, name):
    path = os.path.join(parent, safe_name(name))
    if os.path.isdir(path):
        print(f"ℹ️ Folder already exists, using it: {rel(path)}")
    else:
        os.makedirs(path)
        print(f"📁 Created folder: {rel(path)}")
    return path

def save_bytes(dest_dir, name, data, policy):
    """policy: 'Keep both' | 'Overwrite' | 'Skip'."""
    dst = os.path.join(dest_dir, safe_name(name))
    if os.path.exists(dst):
        if policy.startswith('Skip'):
            print(f"   ⏭️ Skipped (already exists): {name}"); return None
        if policy.startswith('Keep both'):
            dst = unique_path(dst)
    with open(dst, 'wb') as f:
        f.write(data)
    ok = hashlib.md5(data).hexdigest() == md5_file(dst)
    print(f"   {'✅' if ok else '⚠️ CHECKSUM MISMATCH'} {os.path.basename(dst)}  ({human(len(data))})")
    return dst

def unzip_into(zip_file, dest_dir, policy):
    target = os.path.join(dest_dir, os.path.splitext(os.path.basename(zip_file))[0])
    if os.path.exists(target) and policy.startswith('Keep both'):
        target = unique_path(target)
    elif os.path.exists(target) and policy.startswith('Skip'):
        print(f"   ⏭️ Folder '{os.path.basename(target)}' already exists — not unzipped."); return None
    with zipfile.ZipFile(zip_file) as zf:
        members = [m for m in zf.namelist() if not m.startswith('__MACOSX/') and not os.path.basename(m).startswith('._')]
        zf.extractall(target, members)      # extractall strips '..' and absolute paths
    print(f"   📦 Unzipped {len(members)} item(s) → {rel(target)}")
    return target

def upload_from_computer(dest_dir, policy='Keep both', unzip=False):
    os.makedirs(dest_dir, exist_ok=True)
    print(f"📤 Uploading into: {rel(dest_dir)}\n   Click 'Choose Files' below (you can select several).")
    cwd = os.getcwd(); os.chdir(TMP_DIR)          # stray copies Colab writes go to the temp folder
    try:
        picked = colab_files.upload()
    finally:
        os.chdir(cwd)
    saved = []
    for name, data in picked.items():
        dst = save_bytes(dest_dir, name, data, policy)
        if dst:
            saved.append(dst)
            if unzip and ext_of(dst) == '.zip':
                if unzip_into(dst, dest_dir, policy):
                    os.remove(dst)
        try: os.remove(os.path.join(TMP_DIR, name))
        except OSError: pass
    print(f"✅ {len(saved)} file(s) saved to Drive. (Drive syncs in the background — they appear on drive.google.com within a minute.)" if saved else "No files uploaded.")
    return saved

def search_drive(root, text, exact=False, limit=300, seconds=90):
    text = text.strip(); t = text.lower(); res, t0 = [], time.time()
    wild = any(c in text for c in '*?')
    for r, ds, fs in os.walk(root):
        ds.sort()
        for n in ds + fs:
            nl = n.lower()
            hit = fnmatch.fnmatch(nl, t) if wild else (nl == t or (is_native(n) and os.path.splitext(nl)[0] == t)) if exact else (t in nl)
            if hit:
                res.append(os.path.join(r, n))
                if len(res) >= limit: return res, True
        if time.time() - t0 > seconds:
            return res, True
    return res, False

# ───────────────────────── the picker (dropdowns) ─────────────────────────
PICK = {'folder': None, 'selected': []}

def _roots():
    roots = [('My Drive', MY_DRIVE)]
    if os.path.isdir(SHARED_DRIVES):
        roots += [(f'Shared drive: {d}', os.path.join(SHARED_DRIVES, d)) for d in sorted(os.listdir(SHARED_DRIVES))]
    return roots

def _label(p, base=None):
    name = (rel(p) if base == MOUNT_POINT else os.path.relpath(p, base)) if base else os.path.basename(p)
    if os.path.isdir(p):
        return f"📁 {name}/"
    try: size = human(os.path.getsize(p))
    except OSError: size = '?'
    e = ext_of(p)
    if e in NATIVE:         return f"{icon_of(p)} {name}   — {NATIVE[e]['type']} (exported on download)"
    if e in NOT_EXPORTABLE: return f"🚫 {name}   — {NOT_EXPORTABLE[e]} (cannot be downloaded)"
    return f"{icon_of(p)} {name}   — {size}"

class DrivePicker:
    def __init__(self, start):
        L = W.Layout
        self._busy = False
        self.root_dd   = W.Dropdown(options=_roots(), description='Drive:', layout=L(width='340px'))
        self.up_btn    = W.Button(description='⬆ Up', layout=L(width='70px'))
        self.home_btn  = W.Button(description='🏠 Start folder', layout=L(width='130px'))
        self.ref_btn   = W.Button(description='🔄 Refresh', layout=L(width='100px'))
        self.crumb     = W.HTML()
        self.sub_dd    = W.Dropdown(description='📂 Open sub-folder:', style={'description_width': 'initial'}, layout=L(width='560px'))
        self.items     = W.SelectMultiple(options=[], rows=14, layout=L(width='98%'))
        self.all_btn   = W.Button(description='Select all', layout=L(width='100px'))
        self.none_btn  = W.Button(description='Clear', layout=L(width='70px'))
        self.info      = W.HTML()
        self.new_txt   = W.Text(placeholder='e.g. Scrutiny Reports Oct-2026', description='New folder:', layout=L(width='420px'))
        self.new_btn   = W.Button(description='➕ Create & open', button_style='success', layout=L(width='140px'))
        self.find_txt  = W.Text(placeholder='part of a name, e.g. GSTR3B  (wildcards * ? allowed)', description='Search:', layout=L(width='420px'))
        self.find_where= W.Dropdown(options=['in this folder', 'in whole Drive'], layout=L(width='140px'))
        self.find_btn  = W.Button(description='🔍 Search', button_style='info', layout=L(width='100px'))
        self.back_btn  = W.Button(description='↩ Back to folder', button_style='warning', layout=L(width='140px', display='none'))
        self.msg       = W.Output(layout=L(max_height='160px', overflow='auto'))

        self.root_dd.observe(self._on_root, names='value')
        self.sub_dd.observe(self._on_sub, names='value')
        self.items.observe(self._on_items, names='value')
        self.up_btn.on_click(lambda _: self._safe(self.go, os.path.dirname(PICK['folder'])))
        self.home_btn.on_click(lambda _: self._safe(self.go, self.start))
        self.ref_btn.on_click(lambda _: self._safe(self.go, PICK['folder']))
        self.back_btn.on_click(lambda _: self._safe(self.go, PICK['folder']))
        self.all_btn.on_click(lambda _: setattr(self.items, 'value', tuple(v for _, v in self.items.options)))
        self.none_btn.on_click(lambda _: setattr(self.items, 'value', ()))
        self.new_btn.on_click(lambda _: self._safe(self._new_folder))
        self.find_btn.on_click(lambda _: self._safe(self._search))

        self.start = start
        self.go(start)

    # helpers
    def _safe(self, fn, *a):
        with self.msg:
            try: fn(*a)
            except Exception as e: print(f"❌ {type(e).__name__}: {e}")

    def _root_of(self, p):
        best = MY_DRIVE
        for _, r in self.root_dd.options:
            if p.startswith(r) and len(r) >= len(best): best = r
        return best

    def go(self, path):
        path = os.path.normpath(path)
        if not path.startswith(MOUNT_POINT) or os.path.normpath(path) in (MOUNT_POINT, SHARED_DRIVES):
            path = self._root_of(PICK['folder'] or MY_DRIVE)
        if not os.path.isdir(path):
            raise FileNotFoundError(f"Folder not found: {path}")
        self._busy = True
        try:
            PICK['folder'], PICK['selected'] = path, []
            with os.scandir(path) as it:
                entries = sorted(it, key=lambda e: (not e.is_dir(), e.name.lower()))
            dirs = [e.path for e in entries if e.is_dir()]
            self.sub_dd.options = [(f'— choose ({len(dirs)} sub-folder{"s" if len(dirs) != 1 else ""}) —', '')] + [(f'📁 {os.path.basename(d)}', d) for d in dirs]
            self.sub_dd.value = ''
            self.items.options = [(_label(e.path), e.path) for e in entries]
            self.items.value = ()
            self.root_dd.value = self._root_of(path)
            link = folder_link(path)
            self.crumb.value = (f"<div style='font-size:15px;margin:4px 0'><b>📍 {rel(path)}</b>"
                                + (f" &nbsp;<a href='{link}' target='_blank'>open in Drive ↗</a>" if link else '') + "</div>")
            self.back_btn.layout.display = 'none'
            self.up_btn.disabled = path == self._root_of(path)
            self._update_info()
        finally:
            self._busy = False

    def show_results(self, results, title):
        self._busy = True
        try:
            base = PICK['folder'] if all(r.startswith(PICK['folder'] + os.sep) for r in results) else MOUNT_POINT
            self.items.options = [(_label(r, base), r) for r in results]
            self.items.value = ()
            PICK['selected'] = []
            self.crumb.value = f"<div style='font-size:15px;margin:4px 0'><b>🔍 {title}</b> — select, then run cell ④</div>"
            self.back_btn.layout.display = ''
            self._update_info()
        finally:
            self._busy = False

    def _update_info(self):
        sel = PICK['selected']
        if not sel:
            self.info.value = f"<i>{len(self.items.options)} item(s). Click to select · Ctrl/Shift-click for several.</i>"
            return
        size = sum(os.path.getsize(p) for p in sel if os.path.isfile(p) and not is_native(p))
        nd = sum(os.path.isdir(p) for p in sel)
        self.info.value = (f"<b style='color:#1a7f37'>✔ {len(sel)} selected</b> ({human(size)}"
                           + (f" + {nd} folder(s)" if nd else '') + ") — now run cell ④ to download")

    # events
    def _on_root(self, ch):
        if not self._busy: self._safe(self.go, ch['new'])
    def _on_sub(self, ch):
        if not self._busy and ch['new']: self._safe(self.go, ch['new'])
    def _on_items(self, ch):
        if not self._busy:
            PICK['selected'] = list(ch['new']); self._update_info()

    def _new_folder(self):
        p = create_folder(PICK['folder'], self.new_txt.value)
        self.new_txt.value = ''
        self.go(p)

    def _search(self):
        text = self.find_txt.value.strip()
        if not text:
            print("Type part of a file name first."); return
        root = PICK['folder'] if self.find_where.value == 'in this folder' else self._root_of(PICK['folder'])
        print(f"Searching '{text}' in {rel(root)} …")
        res, cut = search_drive(root, text)
        print(f"Found {len(res)} match(es)" + (" (stopped early — refine the search)" if cut else '') + '.')
        self.show_results(res, f"{len(res)} result(s) for '{text}'")

    @property
    def ui(self):
        header = HTML("""<div style="display:flex;justify-content:space-between;align-items:flex-start">
            <div style="font-size:18px;font-weight:600">🗂️ Drive File Manager — picker</div>
            <div style="text-align:right;font-size:11px;color:#666">Created by Ram Lakhan Meena, INSPECTOR CBIC<br>13-Oct-2026 · v2</div></div>""")
        box = W.VBox([
            W.HBox([self.root_dd, self.up_btn, self.home_btn, self.ref_btn]),
            self.crumb, self.sub_dd,
            W.HTML("<b>Files &amp; folders here</b> (select what you want to download):"),
            self.items, W.HBox([self.all_btn, self.none_btn, self.back_btn, self.info]),
            W.HTML("<hr style='margin:6px 0'>"),
            W.HBox([self.new_txt, self.new_btn]),
            W.HBox([self.find_txt, self.find_where, self.find_btn]),
            self.msg])
        return header, box

    def show(self):
        h, b = self.ui
        display(h, b)

start_path = os.path.join(MY_DRIVE, START_FOLDER.strip()) if START_FOLDER.strip() else MY_DRIVE
os.makedirs(start_path, exist_ok=True)
PICKER = DrivePicker(start_path)

display(HTML(f"<div style='padding:8px 12px;border-left:4px solid #1a7f37;background:#f0fff4;color:#111'>"
             f"✅ <b>Ready.</b> Drive connected. Start folder: <b>{rel(start_path)}</b>.<br>"
             f"Next → run cell <b>②</b> to open the picker.</div>"))

## ② Picker — choose folder & files
Use the **Drive** and **Open sub-folder** dropdowns to move around. Click files in the list (Ctrl/Shift-click for several). Create a new folder or search from here too.

In [ ]:
#@title ② PICKER — run, then use the dropdowns { display-mode: "form" }
if 'PICKER' in globals():
    PICKER.show()
else:
    print("⚠️ Run cell ① SETUP first.")

## ③ Upload from your computer → Drive

In [ ]:
#@title ③ UPLOAD files from your computer into Drive { display-mode: "form" }
#@markdown Files go into the folder currently open in the picker (cell ②) — or a new folder inside it.
save_into = "Folder open in the picker"  #@param ["Folder open in the picker", "A NEW folder inside it (type the name below)"]
new_folder_name = ""  #@param {type:"string"}
if_file_already_exists = "Keep both - add (1), (2)"  #@param ["Keep both - add (1), (2)", "Overwrite", "Skip"]
unzip_zip_files_after_upload = False  #@param {type:"boolean"}

if 'PICK' not in globals():
    print("⚠️ Run cell ① SETUP first.")
elif save_into.startswith('A NEW') and not new_folder_name.strip():
    print("⚠️ Type the new folder's name in 'new_folder_name' and run again.")
else:
    _dest = create_folder(PICK['folder'], new_folder_name) if save_into.startswith('A NEW') else PICK['folder']
    upload_from_computer(_dest, if_file_already_exists, unzip_zip_files_after_upload)
    PICKER.go(_dest)   # picker now shows the folder with your uploaded files

## ④ Download from Drive → your computer

In [ ]:
#@title ④ DOWNLOAD to your computer { display-mode: "form" }
#@markdown Select file(s) / folder(s) in the picker (cell ②) first.
what_to_download = "Items selected in the picker"  #@param ["Items selected in the picker", "Entire folder open in the picker (as ZIP)"]
google_docs_sheets_slides_as = "Office (docx / xlsx / pptx)"  #@param ["Office (docx / xlsx / pptx)", "PDF"]
when_several_items = "One ZIP file"  #@param ["One ZIP file", "Separate files"]

if 'PICK' not in globals():
    print("⚠️ Run cell ① SETUP first.")
else:
    _items = [PICK['folder']] if what_to_download.startswith('Entire') else PICK['selected']
    download_items(_items, 'PDF' if google_docs_sheets_slides_as == 'PDF' else 'Office', when_several_items)

## ⑤ Quick download by file name

In [ ]:
#@title ⑤ QUICK DOWNLOAD by name { display-mode: "form" }
#@markdown Type the name (or part of it). Extension is optional. If exactly one file matches it downloads at once; otherwise the matches are loaded into the picker for you to choose.
file_name = ""  #@param {type:"string"}
search_in = "Whole of My Drive"  #@param ["Whole of My Drive", "Folder open in the picker"]
match_type = "Name contains the text"  #@param ["Name contains the text", "Exact name"]
google_files_as = "Office (docx / xlsx / pptx)"  #@param ["Office (docx / xlsx / pptx)", "PDF"]

if not file_name.strip():
    print("⚠️ Type a file name above and run again.")
else:
    _root = MY_DRIVE if search_in.startswith('Whole') else PICK['folder']
    print(f"🔍 Searching '{file_name}' in {rel(_root)} …")
    _res, _cut = search_drive(_root, file_name, exact=match_type == 'Exact name')
    _files = [r for r in _res if os.path.isfile(r)]
    if len(_files) == 1:
        print(f"Found: {rel(_files[0])}")
        download_items(_files, 'PDF' if google_files_as == 'PDF' else 'Office')
    elif not _res:
        print("❌ No match. Check the spelling or try fewer letters.")
    else:
        PICKER.show_results(_res, f"{len(_res)} result(s) for '{file_name}'")
        print(f"Found {len(_res)} matches" + (" (stopped early)" if _cut else "") + " — they are now listed in the picker (cell ②). Select one, then run ④.")

## ⑥ Colab workspace ⇄ Drive
For files produced by other code in this runtime (e.g. `/content/report.xlsx`), or to bring Drive files into `/content` for processing.

In [ ]:
#@title ⑥ COLAB WORKSPACE ⇄ DRIVE { display-mode: "form" }
action = "Copy picker selection → Colab workspace (/content/from_drive)"  #@param ["Copy picker selection → Colab workspace (/content/from_drive)", "Copy a Colab file/folder → folder open in the picker"]
colab_path = "/content/your_file_or_folder"  #@param {type:"string"}
zip_folder_before_copying = False  #@param {type:"boolean"}
google_files_as = "Office (docx / xlsx / pptx)"  #@param ["Office (docx / xlsx / pptx)", "PDF"]

_fmt = 'PDF' if google_files_as == 'PDF' else 'Office'
if action.startswith('Copy picker'):
    os.makedirs(FROM_DRIVE_DIR, exist_ok=True)
    if not PICK['selected']:
        print("⚠️ Select item(s) in the picker first.")
    for _p in PICK['selected']:
        if os.path.isdir(_p):
            _dst = unique_path(os.path.join(FROM_DRIVE_DIR, os.path.basename(_p)))
            shutil.copytree(_p, _dst, ignore=shutil.ignore_patterns(*[f'*{e}' for e in list(NATIVE) + list(NOT_EXPORTABLE)]))
        else:
            _dst = fetch_file(_p, _fmt, FROM_DRIVE_DIR)
        print(f"✅ {rel(_p)}  →  {_dst}")
else:
    _src = colab_path.rstrip('/')
    if not os.path.exists(_src):
        print(f"❌ Not found in Colab: {_src}")
    else:
        if os.path.isdir(_src) and zip_folder_before_copying:
            _src = shutil.make_archive(os.path.join(TMP_DIR, os.path.basename(_src)), 'zip', _src)
        _dst = unique_path(os.path.join(PICK['folder'], os.path.basename(_src)))
        if os.path.isdir(_src):
            shutil.copytree(_src, _dst)
        else:
            shutil.copyfile(_src, _dst)
            print("   checksum", "OK ✅" if md5_file(_src) == md5_file(_dst) else "MISMATCH ⚠️")
        print(f"✅ {_src}  →  {rel(_dst)}")
        PICKER.go(PICK['folder'])

## ⑦ Folder report

In [ ]:
#@title ⑦ FOLDER REPORT (tree of the folder open in the picker) { display-mode: "form" }
levels_deep = 2  #@param {type:"slider", min:1, max:6, step:1}
also_download_full_listing_as_csv = False  #@param {type:"boolean"}

import csv
_root = PICK['folder']; _rows = []; _nf = _nd = _tot = 0
print(f"📍 {rel(_root)}")
for _r, _ds, _fs in os.walk(_root):
    _ds.sort(); _depth = os.path.relpath(_r, _root).count(os.sep) + (0 if _r == _root else 1)
    if _r != _root:
        _nd += 1
        if _depth <= levels_deep: print('   ' * (_depth - 1) + f"📁 {os.path.basename(_r)}/")
    for _f in sorted(_fs):
        _p = os.path.join(_r, _f)
        try: _s = os.path.getsize(_p)
        except OSError: _s = 0
        _nf += 1; _tot += _s
        _rows.append([os.path.relpath(_p, _root), NATIVE.get(ext_of(_f), {}).get('type') or NOT_EXPORTABLE.get(ext_of(_f)) or (ext_of(_f)[1:].upper() or 'file'), _s,
                      datetime.datetime.fromtimestamp(os.path.getmtime(_p)).strftime('%Y-%m-%d %H:%M')])
        if _depth < levels_deep: print('   ' * _depth + f"{icon_of(_p)} {_f}  ({human(_s)})")
print(f"\nTotal: {_nf} files, {_nd} sub-folders, {human(_tot)}")
if also_download_full_listing_as_csv:
    _csv = os.path.join(DL_DIR, f"listing_{os.path.basename(_root)}_{datetime.date.today()}.csv")
    with open(_csv, 'w', newline='', encoding='utf-8-sig') as _fh:
        _w = csv.writer(_fh); _w.writerow(['Path', 'Type', 'Size (bytes)', 'Modified']); _w.writerows(_rows)
    browser_download(_csv)

## ⑧ Finish

In [ ]:
#@title ⑧ DISCONNECT DRIVE (optional — run when completely done) { display-mode: "form" }
#@markdown Makes sure every upload is fully written to Drive before the runtime closes.
confirm_disconnect = False  #@param {type:"boolean"}
if confirm_disconnect:
    drive.flush_and_unmount()
    print("✅ All writes flushed. Drive disconnected — run ① again to reconnect.")
else:
    print("Tick 'confirm_disconnect' and run again to disconnect.")

## ❓ Troubleshooting
* **Downloaded file opens as garbage / is only a few bytes** — that was the v1 bug with Google Docs/Sheets. v2 exports them; make sure you ran ① in this session.
* **Only one of several files downloaded** — your browser blocked multiple downloads. Allow it for `colab.research.google.com`, or choose *One ZIP file*.
* **A popup asks to sign in** — this is needed only once per session for Google Docs/Sheets/Slides export.
* **New upload not visible on drive.google.com** — Drive syncs in the background; wait a minute or run ⑧.
* **"Run cell ① SETUP first"** — the runtime restarted; run ① and ② again.